In [12]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import nibabel as nib
import pandas as pd
import sys
sys.path.append("/home/agustin/phd/synthesis/utils")

import util_freesurfer_segmentation as ufs



from cornucopia import fov, RelabelTransform


In [13]:
files_path = "/home/agustin/phd/synthseg_nextbrain/data/test/inputs/recon_0.2mm"
case_list = os.listdir(files_path)

nextbrain_ontology = pd.read_csv("/home/agustin/phd/synthseg_nextbrain/project/dev/wp1/create_nextbraintree/data/inputs/nextbrain_ontology_v1.csv")
seg_labes = []
for case in case_list:
    print(case)
    case_path = os.path.join(files_path, case)
    img = nib.load(case_path)
    img_data = img.get_fdata()
    img_affine = img.affine

    # print the number of unique values in the image
    unique_values = np.unique(img_data)
    # print(f"Unique values in the image: {len(unique_values)}: \n {unique_values}")
    seg_labes.append(unique_values)

case-3.nii.gz
case-1.nii.gz
case-5.nii.gz
case-2.nii.gz
case-4.nii.gz


In [14]:
# I have list A and list B, and I want to find the elements that are in A but not in B. I can use a list comprehension to do this:

for seg in seg_labes:
    # print(f"Segmentation labels: {seg}")
    missing_labels = [int(x) for x in seg if x not in nextbrain_ontology['allen_id'].values]
    # missing_labels = [int(x) for x in nextbrain_ontology['allen_id'].values if x not in seg]
    print(f"Missing labels: {missing_labels}")

Missing labels: [3, 383]
Missing labels: [3, 383]
Missing labels: [3, 383]
Missing labels: [3, 383]
Missing labels: [3, 383]


In [ ]:
fname_img = "data/inputs/sub-ADNI002S0295_ses-m000_space-MNI_T1w.nii.gz"
fname_seg = "data/inputs/sub-ADNI002S0295_ses-m000_space-MNI_T1wdseg_synthsr.nii.gz"
# npz = np.load(fname)
# dat = torch.as_tensor(npz['vol'][:, :, 112]).T[None]
# lab = torch.as_tensor(npz['seg'][:, :, 112]).T[None]

seg = nib.load(fname_seg).get_fdata()
img = nib.load(fname_img).get_fdata()

thal = np.argwhere(np.isin(seg, [10, 49]))
cx, cy, cz = thal.mean(0).astype(int) if len(thal) else np.array(seg.shape) // 2

seg = ufs.merge_seg96_to_seg3(seg) #map to 0,1,2,3
dat = torch.as_tensor(np.flipud(img[:, :, cz].T).copy())[None]
lab = torch.as_tensor(np.flipud(seg[:, :, cz].T).copy())[None]

lab = RelabelTransform()(lab) #map to 0,1,2,3,...,N


show_image(dat, lab, title='MRI and Labels')

# plt.figure(figsize=(10, 10))
# plt.subplot(1, 2, 1)
# plt.imshow(dat.squeeze(), cmap='gray', interpolation='nearest')
# plt.axis('off')
# plt.title('MRI')
# plt.subplot(1, 2, 2)
# plt.imshow(lab.squeeze(), cmap='tab20', interpolation='nearest')
# plt.axis('off')
# plt.title('Labels')
# plt.show()